# Function Definitions

In [16]:
import pandas as pd
import numpy as np
import tqdm 
from itertools import product
from pathlib import Path


us_state_to_abbrev = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    "Arkansas": "AR",
    "California": "CA",
    "Colorado": "CO",
    "Connecticut": "CT",
    "Delaware": "DE",
    "Florida": "FL",
    "Georgia": "GA",
    "Hawaii": "HI",
    "Idaho": "ID",
    "Illinois": "IL",
    "Indiana": "IN",
    "Iowa": "IA",
    "Kansas": "KS",
    "Kentucky": "KY",
    "Louisiana": "LA",
    "Maine": "ME",
    "Maryland": "MD",
    "Massachusetts": "MA",
    "Michigan": "MI",
    "Minnesota": "MN",
    "Mississippi": "MS",
    "Missouri": "MO",
    "Montana": "MT",
    "Nebraska": "NE",
    "Nevada": "NV",
    "New Hampshire": "NH",
    "New Jersey": "NJ",
    "New Mexico": "NM",
    "New York": "NY",
    "North Carolina": "NC",
    "North Dakota": "ND",
    "Ohio": "OH",
    "Oklahoma": "OK",
    "Oregon": "OR",
    "Pennsylvania": "PA",
    "Rhode Island": "RI",
    "South Carolina": "SC",
    "South Dakota": "SD",
    "Tennessee": "TN",
    "Texas": "TX",
    "Utah": "UT",
    "Vermont": "VT",
    "Virginia": "VA",
    "Washington": "WA",
    "West Virginia": "WV",
    "Wisconsin": "WI",
    "Wyoming": "WY",
    "District of Columbia": "DC",
    "American Samoa": "AS",
    "Guam": "GU",
    "Northern Mariana Islands": "MP",
    "Puerto Rico": "PR",
    "United States Minor Outlying Islands": "UM",
    "U.S. Virgin Islands": "VI",
}
    
# invert the dictionary
abbrev_to_us_state = dict(map(reversed, us_state_to_abbrev.items()))

def list_available_columns(df):
    print(f"Dimensions of data: {df.shape}")
    print(f'optional_cols = [')
    for col in df.columns.to_list():
        print(f"\t\'{col}\',")
    print("\t]")
    
    return None


def default_column_mapping(df):
    print('column_renaming = {')
    for col in df.columns.to_list():
        print(f"\t\'{col}\': \'{col}\',")
    print("\t}")


def return_modified_headers(dictionary):
    updated_columns = {}
    print('column_renaming = {')
    for key, value in dictionary.items():
        if key != value:
            updated_columns[key] = value
            print(f"\t\'{key}\': \'{value}\',")
    print("\t}")

    return updated_columns

def generate_samples_for_averaging(simulation_results, scenario, unique_cols, all_unique_col_combos):
    # returns list of pandas dataframes

    data = simulation_results[simulation_results['Weather Scenario'] == scenario].copy(deep=True)
    samples_dataframes = []

    for combination in tqdm.tqdm(all_unique_col_combos, desc = 'Generating Subsamples for Averaging'):

        datasample = data.copy(deep=True)

        # row_selection = datasample['bldg_id'] > 0   # initialize with all rows
        for i, column in enumerate(unique_cols):
            # row_selection = (row_selection) & (datasample[column] == combination[i])
            datasample = datasample[ datasample[column] == combination[i] ]

        if len(datasample) != 40: 
            # print(datasample['bldg_id'].unique())
            # print(f"sum(row_selection): {len(datasample)}")
            print('Error: See sample data')
            print(datasample)
            print(f'Combination: {combination}')
            raise RuntimeError(f"Error: len(datasample) = {len(datasample)}")

        samples_dataframes.append(datasample)
        # data = data[~row_selection]     # remove the rows that were sampled (to reduce subsequent work)
        
    return samples_dataframes     # returns list of pandas dataframes in the same order as the all_unique_col_combos


def average_samples(samples_dataframes, columns_to_average):
    # returns list of pandas dataframes

    datframes_list = []
    for sample in tqdm.tqdm(samples_dataframes, desc='Averaging Base Scenario Samples'): 

        columns_to_not_average = list(set(sample.columns) - set(columns_to_average))
        metadata = sample[columns_to_not_average].iloc[0, :]                                        # Select the first row of data to not average (using iloc for proper indexing)
        df_averaged = (sample[columns_to_average].mean(axis=0).to_frame()).transpose()              # Calculate the means and store in a new dataframe
        df_averaged = df_averaged.assign(**{col: metadata[col] for col in columns_to_not_average})  # add the metadata back onto the rows
        datframes_list.append(df_averaged)

    return datframes_list       # returns list of pandas dataframes in the same order as the all_unique_col_combos


# def difference_over_mean(simulations, base_scenario, columns_to_take_difference):    
def get_all_combinations(data, unique_cols):
    """
    This function generates all combinations of elements across lists.
    Args:
        lists: A list of lists, where each inner list has the same number of elements.
    Returns:
        A a list containing all combinations as tuples.
    """

    lists = []

    for column in unique_cols: 
        lists.append(data[column].unique())

    # Use product to generate combinations
    return [combination for combination in product(*lists, repeat=1)]


def join(left_data, right_data, how, left_on, right_on, suffixes = ('_l', '_r') ):
    before = len(left_data['in.city'])
    data = pd.merge(left_data, right_data, how=how, left_on=left_on, right_on=right_on, 
                                  left_index=False, right_index=False, sort=False, suffixes=suffixes, copy=None, indicator=False, validate=None)
    after = len(data['in.city'])

    if not after == before: 
        print("Missing data. Data lost")
        print(f"Before: {before}")
        print(f"After: {after}")

        raise RuntimeError()

    return data 


def take_differences(simulation_results, base_scenario_samples_averages, unique_cols, unique_col_value_combos, columns_to_average):
    scenario_differences = []
    fractional_differences = []
    for scenario in (set(simulation_results['Weather Scenario'].unique())): 
            
        print(f'\nTaking Diffferences for Scenario: {scenario}')
        scenario_samples = generate_samples_for_averaging(simulation_results, scenario, unique_cols, unique_col_value_combos)   

        if len(scenario_samples) != len(base_scenario_samples_averages): 
            # print("Error: Unequal number of scenario and base samples")
            raise RuntimeError(f"Unequal number of scenario and base samples")

        for i, sample in enumerate(scenario_samples):
            # because the samples for both the scenarios are generated using the same combinations of unique identitifiers, we can subtract the elements in these lists 
            base_normalized_sample = sample.copy(deep=True)
            fractional_difference = sample.copy(deep=True)
            base_normalized_sample[columns_to_average] = sample[columns_to_average] - base_scenario_samples_averages[i][columns_to_average].iloc[0]
            fractional_difference[columns_to_average] = base_normalized_sample[columns_to_average] / base_scenario_samples_averages[i][columns_to_average].iloc[0]
            scenario_differences.append(base_normalized_sample)
            fractional_differences.append(fractional_difference)

    scenario_differences_df = pd.concat(scenario_differences)
    fractional_differences_df = pd.concat(fractional_differences)
    return [scenario_differences_df, fractional_differences_df]

# Join buildstock metadata onto the simulation results 
### Choose which metadata fields to keep 

In [ ]:
# join buildstock metadata onto the simulation results 
buildstock_file = Path("/Users/camilotoruno/Documents/local_research_data/buildings 24.08.12/buildstock.csv")
buildstock = pd.read_csv(buildstock_file)

print("Buildstock available columns:")
list_available_columns(buildstock)

### Keep desired columns of buildstock metadata
Copy and paste the available optional columns below, then delete any columns you don't want saved with the table

In [18]:
buildstock_keep_columns = [
	'bldg_id',
	'in.sqft',
	'in.ahs_region',
	'in.ashrae_iecc_climate_zone_2004',
	'in.ashrae_iecc_climate_zone_2004_2_a_split',
	'in.bathroom_spot_vent_hour',
	'in.bedrooms',
	'in.building_america_climate_zone',
	'in.cec_climate_zone',
	'in.ceiling_fan',
	'in.census_division',
	'in.census_division_recs',
	'in.census_region',
	'in.city',
	'in.clothes_dryer',
	'in.clothes_washer',
	'in.clothes_washer_presence',
	'in.cooking_range',
	'in.cooling_setpoint',
	'in.cooling_setpoint_has_offset',
	'in.cooling_setpoint_offset_magnitude',
	'in.cooling_setpoint_offset_period',
	'in.corridor',
	'in.county',
	'in.county_and_puma',
	'in.dehumidifier',
	'in.dishwasher',
	'in.door_area',
	'in.doors',
	'in.ducts',
	'in.eaves',
	'in.electric_vehicle',
	'in.federal_poverty_level',
	'in.generation_and_emissions_assessment_region',
	'in.geometry_attic_type',
	'in.geometry_building_horizontal_location_mf',
	'in.geometry_building_horizontal_location_sfa',
	'in.geometry_building_level_mf',
	'in.geometry_building_number_units_mf',
	'in.geometry_building_number_units_sfa',
	'in.geometry_building_type_acs',
	'in.geometry_building_type_height',
	'in.geometry_building_type_recs',
	'in.geometry_floor_area',
	'in.geometry_floor_area_bin',
	'in.geometry_foundation_type',
	'in.geometry_garage',
	'in.geometry_stories',
	'in.geometry_stories_low_rise',
	'in.geometry_story_bin',
	'in.geometry_wall_exterior_finish',
	'in.geometry_wall_type',
	'in.has_pv',
	'in.heating_fuel',
	'in.heating_setpoint',
	'in.heating_setpoint_has_offset',
	'in.heating_setpoint_offset_magnitude',
	'in.heating_setpoint_offset_period',
	'in.holiday_lighting',
	'in.hot_water_distribution',
	'in.hot_water_fixtures',
	'in.hvac_cooling_efficiency',
	'in.hvac_cooling_partial_space_conditioning',
	'in.hvac_cooling_type',
	'in.hvac_has_ducts',
	'in.hvac_has_shared_system',
	'in.hvac_has_zonal_electric_heating',
	'in.hvac_heating_efficiency',
	'in.hvac_heating_type',
	'in.hvac_heating_type_and_fuel',
	'in.hvac_secondary_heating_efficiency',
	'in.hvac_secondary_heating_type_and_fuel',
	'in.hvac_shared_efficiencies',
	'in.hvac_system_is_faulted',
	'in.income',
	'in.income_recs_2015',
	'in.income_recs_2020',
	'in.infiltration',
	'in.interior_shading',
	'in.iso_rto_region',
	'in.lighting',
	'in.lighting_interior_use',
	'in.lighting_other_use',
	'in.location_region',
	'in.mechanical_ventilation',
	'in.misc_extra_refrigerator',
	'in.misc_freezer',
	'in.misc_gas_fireplace',
	'in.misc_gas_grill',
	'in.misc_gas_lighting',
	'in.misc_hot_tub_spa',
	'in.misc_pool',
	'in.misc_pool_heater',
	'in.misc_pool_pump',
	'in.misc_well_pump',
	'in.natural_ventilation',
	'in.neighbors',
	'in.occupants',
	'in.orientation',
	'in.overhangs',
	'in.plug_load_diversity',
	'in.plug_loads',
	'in.puma',
	'in.puma_metro_status',
	'in.pv_orientation',
	'in.pv_system_size',
	'in.radiant_barrier',
	'in.range_spot_vent_hour',
	'in.reeds_balancing_area',
	'in.refrigerator',
	'in.roof_material',
	'in.schedules',
	'in.solar_hot_water',
	'in.state',
	'in.tenure',
	'in.units_represented',
	'in.usage_level',
	'in.vacancy_status',
	'in.vintage',
	'in.vintage_acs',
	'in.water_heater_efficiency',
	'in.water_heater_fuel',
	'in.water_heater_in_unit',
	'in.weather_file_city',
	'in.weather_file_latitude',
	'in.weather_file_longitude',
	'in.window_areas',
	'in.windows',
	]

buildstock = buildstock[buildstock_keep_columns]

### Choose the columns from the simulation results to keep 
Copy and paste the available optional columns below, then delete any columns you don't want. You can reorganize the list of columns if you'd like to reorganize them. 


In [ ]:
simulation_file = Path("/Users/camilotoruno/Documents/local_research_data/simulations 24.08.12/simulations.csv")
simulation_results = pd.read_csv(simulation_file)

print("Simulations results available columns:")
list_available_columns(simulation_results)

In [20]:
optional_cols = [
	'bldg_id',
	'Year',
	'Month',
	'Weather Scenario',

	'Environment:Site Outdoor Air Drybulb Temperature [C](Monthly)',
	'Environment:Site Outdoor Air Wetbulb Temperature [C](Monthly)',

	'Heating:NaturalGas [J](Monthly)',
	'Heating:Electricity [J](Monthly)',
	'Heating:DistrictHeating [J](Monthly)',
	'Heating:Propane [J](Monthly)',
	'Heating:FuelOilNo2 [J](Monthly)',
    
	'Cooling:Electricity [J](Monthly)',
	'Cooling:DistrictCooling [J](Monthly)',

	'Electricity:Facility [J](Monthly)',
	'ElectricityPurchased:Facility [J](Monthly)',
	'ElectricitySurplusSold:Facility [J](Monthly)',
	'ElectricityNet:Facility [J](Monthly)',

	'DistrictHeating:Facility [J](Monthly)',
	'DistrictCooling:Facility [J](Monthly)',

	'Propane:Facility [J](Monthly)',
	'FuelOilNo2:Facility [J](Monthly)',
	'NaturalGas:Facility [J](Monthly)',
	]

simulation_results = simulation_results[optional_cols]

### Join buildstock onto simulations

In [21]:
integer_columns = [	
    "bldg_id",
	"Year",
	"Month",
    ]

simulation_results['Month'] = simulation_results['Month'].str.strip()		# strip empty space from month strings

import calendar
month_dictionary = {month: index for index, month in enumerate(calendar.month_name) if month}	# create month name to number dictionary - use Month: Month number (e.g. October: 10)
# month_dictionary = {month: calendar.month_abbr[index] for index, month in enumerate(calendar.month_name) if month}	# create month name to number dictionary - use Month: Abbrevation (e.g. October: Oct)
if sum(simulation_results['Month'].isin(list(month_dictionary.keys()))) > 0:
    simulation_results['Month'] = simulation_results['Month'].map(month_dictionary)   			# map the month name to month number 
    

simulation_results[integer_columns] = simulation_results[integer_columns].astype(int)
simulation_results = pd.merge(simulation_results, buildstock, how='left', on="bldg_id", 
                                  left_index=False, right_index=False, sort=False, suffixes=('', '_y'), copy=None, indicator=False, validate=None)

simulation_results = simulation_results.drop(simulation_results.filter(regex='_y$').columns, axis=1)  # https://stackoverflow.com/questions/19125091/pandas-merge-how-to-avoid-duplicating-columns


# Average latitude & longitude for each city
ResStock has multiple values for lat/long for each city. Average these results for visualization / analysis

In [22]:
for city in simulation_results['in.city'].unique():
    city_rows = simulation_results['in.city'] == city 
    simulation_results.loc[city_rows, 'in.weather_file_latitude'] = simulation_results.loc[city_rows, 'in.weather_file_latitude'].mean()
    simulation_results.loc[city_rows, 'in.weather_file_longitude'] = simulation_results.loc[city_rows, 'in.weather_file_longitude'].mean()


# Cost Calculations
#### Note ** If pricing data not yet averaged and tabularized for joining onto simulation, perform that using pricing_data_cleaning.ipynb
Propane pricing missing for some state. Use US pricing instead. Enter the state codes that need US prices (no state pricing)

In [23]:
states_to_use_national_propane_prices = ['OR', 'CA', 'AZ', 'NM']

### Join the pricing data onto simulation results 

In [ ]:
class obj:
    def __init__(self) -> None:
        pass

files = obj()
files.pricing_directory = Path("/Volumes/seas-mtcraig/data_sharing/Energy Burdens Under Climate Change/Energy rates")
files.natural_gas = Path(files.pricing_directory, "Natural gas/natural_gas_pricing.csv")
files.fuel_oil_no2 = Path(files.pricing_directory, "Oil/average_monthly_fuel_oil_prices.csv") 
files.elec = Path(files.pricing_directory, "Electricity/EIA/EIA_Average_retail_price_of_electricity.csv")
files.propane = Path(files.pricing_directory, "Propane/monthly_average_propane_price_final.csv") 

propane_pricing = pd.read_csv(files.propane)
electricity_pricing = pd.read_csv(files.elec)
fuel_oil_no2_pricing = pd.read_csv(files.fuel_oil_no2)
natural_gas_pricing = pd.read_csv(files.natural_gas)

# Join the electricity pricing on the simulation data 
simulation_results['in.city'] = simulation_results['in.city'].str.split(', ').str[1]

print("Joining Electricity Pricing")
# electricity_pricing['State (temporary)'] = electricity_pricing['Region']
simulation_results = join(simulation_results, electricity_pricing, how='inner', left_on=['in.state', 'Month'], right_on=['Region', 'Month'], suffixes=['', '_y'])
simulation_results = simulation_results.drop(simulation_results.filter(regex='_y$').columns, axis=1)  # https://stackoverflow.com/questions/19125091/pandas-merge-how-to-avoid-duplicating-columns

# Join the natural gas pricing on the simulation data
print("Joining Natural Gas Pricing")
simulation_results = join(simulation_results, natural_gas_pricing, how='inner', left_on=['in.state', 'Month'], right_on=['State', 'Month'], suffixes=['', '_y'])
simulation_results = simulation_results.drop(simulation_results.filter(regex='_y$').columns, axis=1)  # https://stackoverflow.com/questions/19125091/pandas-merge-how-to-avoid-duplicating-columns

# Join the fuel_oil_no2_pricing pricing on the simulation data
print("Joining Fuel Oil Pricing")
simulation_results = join(simulation_results, fuel_oil_no2_pricing, how='inner', left_on=['Month'], right_on=['Month'], suffixes=['', '_y'])
simulation_results = simulation_results.drop(simulation_results.filter(regex='_y$').columns, axis=1)  # https://stackoverflow.com/questions/19125091/pandas-merge-how-to-avoid-duplicating-columns

# Join the propoane pricing on the simulation data 
print("Joining Propane Pricing")
simulation_results['State (Temporary)'] = simulation_results['in.state']
simulation_results.loc[ simulation_results['State (Temporary)'].isin(states_to_use_national_propane_prices), 'State (Temporary)' ] = 'U.S.'   # change to US for joining US pricing on states without pricing data
simulation_results = join(simulation_results, propane_pricing, how='inner', left_on=['State (Temporary)', 'Month'], right_on=['State', 'Month'], suffixes=['', '_y'])
simulation_results = simulation_results.drop(simulation_results.filter(regex='_y$').columns, axis=1)  # https://stackoverflow.com/questions/19125091/pandas-merge-how-to-avoid-duplicating-columns
simulation_results = simulation_results.drop(columns=['State (Temporary)'])  # drop temporary state column 

### Unit Conversions

In [25]:
# print([col for col in simulation_results.columns])

# propane_states = [state for state in propane_pricing['State'].unique()]
# res_states = [state for state in simulation_results['State (Temporary)'].unique()]
# for state in res_states:
#     if state not in propane_states:
#         print(state)

In [26]:
# Unit coversions 
j_to_kwh = 3600000 # Joule / kWh electricity https://www.rapidtables.com/convert/energy/Joule_to_kWh.html

# natural gas 
Gj_to_Mcf = 1.0551 # https://www.naturalgasintel.com/natural-gas-converter/
j_to_Mcf = 1e9 * Gj_to_Mcf  

# Propane 
propane_btu_per_gallon = 91452 # Btu #  https://www.eia.gov/energyexplained/units-and-calculators/
j_per_btu = 1055.05585262 # J / BTU
j_per_gallon_propane = propane_btu_per_gallon * j_per_btu

# number 2 fuel oil   
j_per_gallon_no2_fuel_oil = 146520000      # https://www.convertunits.com/from/gallon+[U.S.]+of+distillate+no.+2+fuel+oil/to/joule

new_columns = obj()

# Electricity 
simulation_results['Cooling:Electricity [kWh](Monthly)'] = simulation_results['Cooling:Electricity [J](Monthly)'] / j_to_kwh
simulation_results['Heating:Electricity [kWh](Monthly)'] = simulation_results['Heating:Electricity [J](Monthly)'] / j_to_kwh
simulation_results['ElectricityPurchased:Facility [kWh](Monthly)'] = simulation_results['ElectricityPurchased:Facility [J](Monthly)'] / j_to_kwh

# Natural Gas
simulation_results['Heating:NaturalGas [Mcf](Monthly)'] = simulation_results['Heating:NaturalGas [J](Monthly)'] / j_to_Mcf
simulation_results['NaturalGas:Facility [Mcf](Monthly)'] = simulation_results['NaturalGas:Facility [J](Monthly)'] / j_to_Mcf

# Propane
simulation_results['Heating:Propane [Gal](Monthly)'] = simulation_results['Heating:Propane [J](Monthly)'] / j_per_gallon_propane
simulation_results["Propane:Facility [Gal](Monthly)"] = simulation_results['Propane:Facility [J](Monthly)'] / j_per_gallon_propane

# Fuel oil 
simulation_results['Heating:FuelOilNo2 [Gal](Monthly)'] = simulation_results['Heating:FuelOilNo2 [J](Monthly)'] / j_per_gallon_no2_fuel_oil
simulation_results["FuelOilNo2:Facility [Gal](Monthly)"] = simulation_results["FuelOilNo2:Facility [J](Monthly)"] / j_per_gallon_no2_fuel_oil

### Cost Calculuations

In [27]:
# Electricity
simulation_results['Cost Cooling:Electricity [$](Monthly)'] = simulation_results['Cooling:Electricity [kWh](Monthly)'] * simulation_results["Price (cents per kelowatthour)"] / 100
simulation_results['Cost Heating:Electricity [$](Monthly)'] = simulation_results['Heating:Electricity [kWh](Monthly)'] * simulation_results["Price (cents per kelowatthour)"] / 100
simulation_results['Cost ElectricityPurchased:Facility [$](Monthly)'] = simulation_results['ElectricityPurchased:Facility [kWh](Monthly)'] * simulation_results["Price (cents per kelowatthour)"] / 100

# Natural gas 
simulation_results['Cost Heating:NaturalGas [$](Monthly)'] = simulation_results['Heating:NaturalGas [Mcf](Monthly)'] * simulation_results["Price of Natural Gas Delivered to Residential Consumers (Dollars per Thousand Cubic Feet)"]
simulation_results['Cost NaturalGas:Facility [$](Monthly)'] = simulation_results['NaturalGas:Facility [Mcf](Monthly)'] * simulation_results["Price of Natural Gas Delivered to Residential Consumers (Dollars per Thousand Cubic Feet)"]

# Propane
simulation_results['Cost Heating:Propane [$](Monthly)'] = simulation_results['Heating:Propane [Gal](Monthly)'] * simulation_results['Monthly U.S. Propane Residential Price (Dollars per Gallon)']
simulation_results["Cost Propane:Facility [$](Monthly)"] = simulation_results["Propane:Facility [Gal](Monthly)"] * simulation_results['Monthly U.S. Propane Residential Price (Dollars per Gallon)']

# Fuel oil 
simulation_results['Cost Heating:FuelOilNo2 [$](Monthly)'] = simulation_results['Heating:FuelOilNo2 [Gal](Monthly)'] * simulation_results['Monthly No. 2 Heating Oil Residential Price Dollars per Gallon']
simulation_results["Cost FuelOilNo2:Facility [Gal](Monthly)"] = simulation_results["FuelOilNo2:Facility [Gal](Monthly)"] * simulation_results['Monthly No. 2 Heating Oil Residential Price Dollars per Gallon']

### Define which columns to sum for total heating and cooling costs & energy usage

In [ ]:
list_available_columns(simulation_results)

In [ ]:
heating_cols =	[
	"Cost Heating:Electricity [$](Monthly)",
	"Cost Heating:NaturalGas [$](Monthly)",
	"Cost Heating:Propane [$](Monthly)",
	"Cost Heating:FuelOilNo2 [$](Monthly)",
    ]

cooling_cols =	[
	"Cost Cooling:Electricity [$](Monthly)",
    ]

facility_energy_costs_cols = [
	'Cost ElectricityPurchased:Facility [$](Monthly)',
	'Cost NaturalGas:Facility [$](Monthly)',
	'Cost Propane:Facility [$](Monthly)',
	'Cost FuelOilNo2:Facility [Gal](Monthly)',
]

heating_energy_usage_cols = [
	'Heating:NaturalGas [J](Monthly)',
	'Heating:Electricity [J](Monthly)',
	'Heating:DistrictHeating [J](Monthly)',
	'Heating:Propane [J](Monthly)',
	'Heating:FuelOilNo2 [J](Monthly)',
	]

facility_energy_usage_columns = [
	'Electricity:Facility [J](Monthly)',
	'Propane:Facility [J](Monthly)',
	'FuelOilNo2:Facility [J](Monthly)',
	'NaturalGas:Facility [J](Monthly)',
]

cooling_energy_usage_cols = ['Cooling:Electricity [J](Monthly)']

total_cost_columns = ['Total Cost Heating [$](Monthly)', 
                      'Total Cost Cooling [$](Monthly)']

simulation_results['Total Cost Heating [$](Monthly)'] = simulation_results[heating_cols].fillna(0).sum(axis=1)      # replace NaN values with zero and sum all heating columns
simulation_results['Total Cost Cooling [$](Monthly)']  = simulation_results[cooling_cols].fillna(0).sum(axis=1)		# replace NaN values with zero and sum all cooling columns



simulation_results['Total Cost Space Conditioning [$](Monthly)'] = simulation_results[total_cost_columns].sum(axis=1) 
simulation_results['Cost Energy:Facility [$](Monthly)'] = simulation_results[facility_energy_costs_cols].sum(axis=1)

simulation_results['Total Energy Usage:Heating [kWh](Monthly)'] = simulation_results[heating_energy_usage_cols].sum(axis=1) / j_to_kwh
simulation_results['Total Energy Usage:Cooling [kWh](Monthly)'] = simulation_results[cooling_energy_usage_cols].sum(axis=1) / j_to_kwh
simulation_results['Total Energy Usage:Space Conditioning [kWh](Monthly)'] = simulation_results[heating_energy_usage_cols + cooling_energy_usage_cols].sum(axis=1) / j_to_kwh
simulation_results['Total Energy Usage:Facility [kWh](Monthly)'] = simulation_results[facility_energy_usage_columns].sum(axis=1) / j_to_kwh


""" 
print(sum(simulation_results['Total Cost Space Conditioning [$](Monthly)'] > 0) / len(simulation_results))

Why do some months have zero total space conditioning cost  	?????????????????????
	- Cost of district heating/cooling not accounted for?
"""

### Energy Burdens Calculations
Also Calculate More Granular Income Bins

In [30]:
# Convert annual income bins to ranges
# Split the column on the delimiter - or < or >, replace empty cells with 0, and cast as integer
simulation_results[["Income - Low [Annual]", "Income - High [Annual]"]] = simulation_results["in.income"].str.split(expand=True, pat='[-<>]').replace('', 0).astype(int) 

simulation_results['in.occupants'].replace({"10+": "10"}, inplace=True)
simulation_results['in.occupants'] = simulation_results['in.occupants'].astype(int)

# High monthly income = high end of income / 12 
simulation_results[["Income - Low [Monthly]", "Income - High [Monthly]"]] = simulation_results[["Income - Low [Annual]", "Income - High [Annual]"]] / 12 
# Calculate More Granular Income - Normalize income by number of residents
simulation_results["Income per Occupant -  High [Annual]"] = simulation_results["Income - High [Annual]"] / simulation_results["in.occupants"]

# High income energy burden = total monthly energy costs / high monthly income 
# Low income energy burden = total monthly energy costs / low monthly income 
simulation_results['Energy Burden - High [Monthly]']                    = simulation_results['Cost Energy:Facility [$](Monthly)'] / simulation_results["Income - High [Monthly]"]
simulation_results["Space Conditioning Energy Burden - Low [Monthly]"]  = simulation_results['Total Cost Space Conditioning [$](Monthly)'] / simulation_results["Income - Low [Monthly]"]
simulation_results["Space Conditioning Energy Burden - High [Monthly]"] = simulation_results['Total Cost Space Conditioning [$](Monthly)'] / simulation_results["Income - High [Monthly]"]
simulation_results['Cooling Energy Burden - High [Monthly]']            = simulation_results['Total Cost Cooling [$](Monthly)'] / simulation_results["Income - High [Monthly]"]
simulation_results['Heating Energy Burden - High [Monthly]']            = simulation_results['Total Cost Heating [$](Monthly)'] / simulation_results["Income - High [Monthly]"]
simulation_results = simulation_results.replace([np.inf, -np.inf, np.nan], 0)   # replace infinity, and not a number(nan) values with zeros - these are due to null or zero values for income low or high end of the range

# Merge the global warming level from TGW onto the data

In [31]:
global_warming_signals = pd.read_csv(Path("/Volumes/seas-mtcraig/data_sharing/Energy Burdens Under Climate Change/TGW Warming Deltas/Global Annual Deltas/Global_Annual_Delta_long_table.csv"))
conus_monthly_warming_signals = pd.read_csv(Path("/Volumes/seas-mtcraig/data_sharing/Energy Burdens Under Climate Change/TGW Warming Deltas/CONUS_Monthly_Deltas/CONUS_Monthly_Delta_all_scenarios.csv"))
conus_annual_warming_signals = pd.read_csv(Path("/Volumes/seas-mtcraig/data_sharing/Energy Burdens Under Climate Change/TGW Warming Deltas/CONUS_Annual_Delta.csv"))

conus_monthly_warming_signals['Month'] = conus_monthly_warming_signals['Month'].map(month_dictionary)   			# map the month name to month number 

simulation_results['Weather Scenario'] = simulation_results['Weather Scenario'].str.split('_').str[0]  # rename simulation results scenarios to remove date range from scenario name (years held in a different column)

# join annual global climate change signal 
simulation_results = pd.merge(simulation_results, global_warming_signals, how='left', left_on=['Weather Scenario', 'Year'], right_on=['Weather Scenario', 'Year'], 
                                  left_index=False, right_index=False, sort=False, suffixes=('', '_y'), copy=None, indicator=False, validate=None)

simulation_results = simulation_results.drop(simulation_results.filter(regex='_y$').columns, axis=1)  # https://stackoverflow.com/questions/19125091/pandas-merge-how-to-avoid-duplicating-columns

# join monthly CONUS climate change signal
simulation_results = pd.merge(simulation_results, conus_monthly_warming_signals, how='left', left_on=['Weather Scenario', 'Year', 'Month'], right_on=['Weather Scenario', 'Year', 'Month'], 
                                  left_index=False, right_index=False, sort=False, suffixes=('', '_y'), copy=None, indicator=False, validate=None)

simulation_results = simulation_results.drop(simulation_results.filter(regex='_y$').columns, axis=1)  # https://stackoverflow.com/questions/19125091/pandas-merge-how-to-avoid-duplicating-columns

# join the annual CONUS climate change signal 
simulation_results = pd.merge(simulation_results, conus_annual_warming_signals, how='left', left_on=['Weather Scenario', 'Year'], right_on=['Weather Scenario', 'Year'], 
                                  left_index=False, right_index=False, sort=False, suffixes=('', '_y'), copy=None, indicator=False, validate=None)

simulation_results = simulation_results.drop(simulation_results.filter(regex='_y$').columns, axis=1)  # https://stackoverflow.com/questions/19125091/pandas-merge-how-to-avoid-duplicating-columns

Drop duplicate columns

In [32]:
# drop duplicate columns 
duplicate_cols = simulation_results.columns[simulation_results.columns.duplicated()]
simulation_results.drop(columns=duplicate_cols, inplace=True)

# Take differences across scenarios

In [ ]:
list_available_columns(simulation_results)

### Define columns to take differences between scenarios 

In [34]:
columns_to_take_difference = [
	'Environment:Site Outdoor Air Drybulb Temperature [C](Monthly)',
	'Environment:Site Outdoor Air Wetbulb Temperature [C](Monthly)',
    
	'Cooling:Electricity [kWh](Monthly)',
	'Heating:Electricity [kWh](Monthly)',
	'Heating:NaturalGas [Mcf](Monthly)',
	'Heating:Propane [Gal](Monthly)',
	'Heating:FuelOilNo2 [Gal](Monthly)',
    
	'Propane:Facility [Gal](Monthly)',
	'NaturalGas:Facility [Mcf](Monthly)',
	'ElectricityPurchased:Facility [kWh](Monthly)',
	'FuelOilNo2:Facility [Gal](Monthly)',
    
	'Cost Cooling:Electricity [$](Monthly)',
	'Cost Heating:Electricity [$](Monthly)',
	'Cost ElectricityPurchased:Facility [$](Monthly)',
	'Cost Heating:NaturalGas [$](Monthly)',
	'Cost NaturalGas:Facility [$](Monthly)',
	'Cost Heating:Propane [$](Monthly)',
	'Cost Propane:Facility [$](Monthly)',
	'Cost Heating:FuelOilNo2 [$](Monthly)',
	'Cost FuelOilNo2:Facility [Gal](Monthly)',
    
	'Total Cost Heating [$](Monthly)',
	'Total Cost Cooling [$](Monthly)',
	'Total Cost Space Conditioning [$](Monthly)',
	'Cost Energy:Facility [$](Monthly)',
    
	'Total Energy Usage:Heating [kWh](Monthly)',
    'Total Energy Usage:Cooling [kWh](Monthly)',
	'Total Energy Usage:Space Conditioning [kWh](Monthly)',
    'Total Energy Usage:Facility [kWh](Monthly)',
    
	'Space Conditioning Energy Burden - Low [Monthly]',
	'Space Conditioning Energy Burden - High [Monthly]',
    'Cooling Energy Burden - High [Monthly]',
	'Heating Energy Burden - High [Monthly]',
	'Energy Burden - High [Monthly]',
    ]

metadata_columns = list(set(simulation_results.columns) - set(columns_to_take_difference))

base_scenario = "historical"
unique_cols = ['bldg_id', 'Month', 'Year', 'Weather Scenario']

# take the average of a subset of columns and where a set of different key columns match a certain value
simulation_results[columns_to_take_difference] = simulation_results[columns_to_take_difference].replace(['', np.nan], 0).astype(float)

## Take differences and calculate the fractional changes in metrics of interest

# Remove this **CRUTCH** 
Filters out data to sidestep issue with missing weather year EPW for 2043 from Louisville RCP 4.5 hotter. 

In [35]:

# ########################################################################################################################
# ########################        ########################        ########################        ########################
# ########################################################################################################################
# ########################        ########################        ########################        ########################
# ########################################################################################################################

# # REMOVE THIS SECTION ONCE DATA ISSUE RESOLVED

# # need to run EPW generation for 2043 from Louisville RCP 4.5 hotter. 
# year_to_remove = 23
# city_to_modify = "Louisville Jefferson County Metro Government Balance"
# remove_years = []
# scenarios_to_remove_values_from = ['historical', 'rcp85cooler']
# for scenario in scenarios_to_remove_values_from:
#     # remove the last year from that scenario 
#     scenario_data = simulation_results[simulation_results['Weather Scenario'] == scenario]
#     city_data = scenario_data[scenario_data['in.city'] == city_to_modify]
#     years = list(city_data['Year'].unique())
#     years.sort()   # sort the years
#     remove_years.append(years[year_to_remove])

# filter = (simulation_results["in.city"] == city_to_modify) & \
#         (simulation_results['Year'].isin(remove_years))


# # print(simulation_results[filter])

# # print(len(simulation_results))

# simulation_results = simulation_results[~filter].reset_index(drop=True)

# # print(len(simulation_results))

# ########################################################################################################################
# ########################        ########################        ########################        ########################
# ########################################################################################################################
# ########################        ########################        ########################        ########################
# ########################################################################################################################

In [36]:
# simulation_results.loc[filter, 'Year']
# print(remove_years)

In [37]:
simulation_results = simulation_results.sort_values(by=unique_cols)
base_scenario_data = simulation_results[simulation_results['Weather Scenario'] == base_scenario].copy(deep=True).reset_index(drop=True)

scenario_differences_dataframes = []
scenario_fractional_differences_dataframes = []

# for each scenario (aside from the base scenario)
for scenario in list( set(simulation_results['Weather Scenario'].unique()) - set([base_scenario]) ):

    scenario_data = simulation_results[simulation_results['Weather Scenario'] == scenario].copy(deep=True).reset_index(drop=True)


    # Check for errors and print info 
    if len(scenario_data) != len(base_scenario_data): 
        base_cities = base_scenario_data['in.city'].unique()
        scenario_cities = scenario_data['in.city'].unique()
        
        if set(base_cities) != set(scenario_cities): print('Sets of cities are different')

        for city in scenario_cities:
            base_count = sum(base_scenario_data['in.city'] == city)
            scenario_count = sum(scenario_data['in.city'] == city)
            if base_count != scenario_count: 
                print(f"\nCity: {city}")
                print(f"Base count: {base_count}")
                print(f"Scenario count: {scenario_count}")

        raise RuntimeError(f"Scenario {scenario} length: {len(scenario_data)}. Base weather scenario length {len(base_scenario_data)}")

    scenario_difference = scenario_data[metadata_columns].copy(deep=True)
    scenario_fractional_difference = scenario_data[metadata_columns].copy(deep=True)

    scenario_difference[columns_to_take_difference] = scenario_data[columns_to_take_difference] - base_scenario_data[columns_to_take_difference]    # calc. absolute differences  
    scenario_fractional_difference[columns_to_take_difference] = scenario_difference[columns_to_take_difference] / base_scenario_data[columns_to_take_difference]  # calc. fractional differences
    
    # append the calculated dataframes to list of dataframes
    scenario_differences_dataframes.append(scenario_difference)
    scenario_fractional_differences_dataframes.append(scenario_fractional_difference)

scenario_differences            = pd.concat(scenario_differences_dataframes,            ignore_index=True)
scenario_fractional_differences = pd.concat(scenario_fractional_differences_dataframes, ignore_index=True)

In [ ]:
simulation_results  # Show table preview

## Choose Columns to Save

In [ ]:
list_available_columns(simulation_results)

In [40]:
save_cols = [    
	'bldg_id',
	'Year',
	'Month',
	'Weather Scenario',

	'in.ashrae_iecc_climate_zone_2004',
	'in.building_america_climate_zone',
	'in.city',
	'in.cooling_setpoint',		
	'in.federal_poverty_level',
	'in.geometry_building_type_acs',
	'in.geometry_building_type_recs',
	'in.geometry_stories',
	'in.geometry_wall_type',
	'in.heating_fuel',
	'in.hvac_heating_type',
	'in.heating_setpoint',
	'in.hvac_cooling_efficiency',
	'in.hvac_cooling_partial_space_conditioning',
	'in.hvac_cooling_type',
	'in.hvac_has_shared_system',
	'in.hvac_heating_efficiency',
	'in.income',
	'in.infiltration',
	'in.occupants',
	'in.state',
	'in.tenure',
	'in.vacancy_status',
	'in.vintage_acs',
	'in.weather_file_latitude',
	'in.weather_file_longitude',
	'in.windows',
    
    'Income per Occupant -  High [Annual]',
	'Price (cents per kelowatthour)',
	'Price of Natural Gas Delivered to Residential Consumers (Dollars per Thousand Cubic Feet)',
	'Monthly No. 2 Heating Oil Residential Price Dollars per Gallon',
	'Monthly U.S. Propane Residential Price (Dollars per Gallon)',

	'Environment:Site Outdoor Air Drybulb Temperature [C](Monthly)',
	'Environment:Site Outdoor Air Wetbulb Temperature [C](Monthly)',
	'Heating:NaturalGas [J](Monthly)',
	'Heating:Electricity [J](Monthly)',
	'Heating:DistrictHeating [J](Monthly)',
	'Heating:Propane [J](Monthly)',
	'Heating:FuelOilNo2 [J](Monthly)',
	'Cooling:Electricity [J](Monthly)',
	'Cooling:DistrictCooling [J](Monthly)',
    
	'Cost Cooling:Electricity [$](Monthly)',
	'Cost Heating:Electricity [$](Monthly)',
	'Cost Heating:NaturalGas [$](Monthly)',
	'Cost Heating:Propane [$](Monthly)',
	'Cost Heating:FuelOilNo2 [$](Monthly)',
	'Cost Energy:Facility [$](Monthly)',
	'Total Cost Heating [$](Monthly)',
	'Total Cost Cooling [$](Monthly)',
	'Total Cost Space Conditioning [$](Monthly)',
    
	'Total Energy Usage:Heating [kWh](Monthly)',
    'Total Energy Usage:Cooling [kWh](Monthly)',
	'Total Energy Usage:Space Conditioning [kWh](Monthly)',
    'Total Energy Usage:Facility [kWh](Monthly)',

	'Energy Burden - High [Monthly]',
	'Space Conditioning Energy Burden - Low [Monthly]',
	'Space Conditioning Energy Burden - High [Monthly]',
	'Cooling Energy Burden - High [Monthly]',
	'Heating Energy Burden - High [Monthly]',

	'Global Climate Change [Celsius]',
	'CONUS Monthly Climate Change [Celsius]',
    'CONUS Annual Climate Change [Celsius]',
	]

simulation_results = simulation_results[save_cols]
scenario_differences = scenario_differences[save_cols]
scenario_fractional_differences = scenario_fractional_differences[save_cols]

# Calculate proxy for AC demand

In [ ]:
differences_with_ac_demand = []
for city in scenario_differences['in.city'].unique():

    city = scenario_differences[scenario_differences['in.city'] == city].copy(deep=True)

    boolean_have_ac = city['in.hvac_cooling_type'] != 0
    AC_penetration = sum( boolean_have_ac ) / len(city)

    print(AC_penetration)

    city = city[boolean_have_ac]
    city['AC Demand'] = (1 - AC_penetration) * city['Cooling:Electricity [J](Monthly)'] 
    differences_with_ac_demand.append(city)

differences_with_ac_demand = pd.concat(differences_with_ac_demand, ignore_index=True)

## Put all data into one table (with different naming for abs/fractional diff/abs differences)

In [42]:
columnds_to_name_by_calculation = [
    'Environment:Site Outdoor Air Drybulb Temperature [C](Monthly)',
	'Environment:Site Outdoor Air Wetbulb Temperature [C](Monthly)',
	'Heating:NaturalGas [J](Monthly)',
	'Heating:Electricity [J](Monthly)',
	'Cooling:Electricity [J](Monthly)',
	'Heating:DistrictHeating [J](Monthly)',
	'Cooling:DistrictCooling [J](Monthly)',
	'Heating:Propane [J](Monthly)',
	'Heating:FuelOilNo2 [J](Monthly)',
	'Cost Cooling:Electricity [$](Monthly)',
	'Cost Heating:Electricity [$](Monthly)',
	'Cost Heating:NaturalGas [$](Monthly)',
	'Cost Heating:Propane [$](Monthly)',
	'Cost Heating:FuelOilNo2 [$](Monthly)',
	'Cost Energy:Facility [$](Monthly)',
	'Total Cost Heating [$](Monthly)',
	'Total Cost Cooling [$](Monthly)',
	'Total Cost Space Conditioning [$](Monthly)',
    
	'Total Energy Usage:Heating [kWh](Monthly)',
	'Total Energy Usage:Cooling [kWh](Monthly)',
	'Total Energy Usage:Space Conditioning [kWh](Monthly)',
	'Total Energy Usage:Facility [kWh](Monthly)',

	'Energy Burden - High [Monthly]',
	'Space Conditioning Energy Burden - Low [Monthly]',
	'Space Conditioning Energy Burden - High [Monthly]',
	'Cooling Energy Burden - High [Monthly]',
	'Heating Energy Burden - High [Monthly]',
    ]

# Rename the difference columns 
abs_diff_cols = ['Abs diff in ' + col for col in columnds_to_name_by_calculation]
frac_diff_cols = ['Frac diff in ' + col for col in columnds_to_name_by_calculation]

abs_column_renaming = {columnds_to_name_by_calculation[i]: abs_diff_cols[i] for i in range(len(columnds_to_name_by_calculation))}
frac_column_renaming = {columnds_to_name_by_calculation[i]: frac_diff_cols[i] for i in range(len(columnds_to_name_by_calculation))}

scenario_differences = scenario_differences.rename(columns=abs_column_renaming, errors="raise")
scenario_fractional_differences = scenario_fractional_differences.rename(columns=frac_column_renaming, errors="raise")


# join absolute differences
simulation_results = pd.merge(simulation_results, scenario_differences, how='left', left_on=unique_cols, right_on=unique_cols, 
                                  left_index=False, right_index=False, sort=False, suffixes=('', '_y'), copy=None, indicator=False, validate=None)

simulation_results = simulation_results.drop(simulation_results.filter(regex='_y$').columns, axis=1)  # https://stackoverflow.com/questions/19125091/pandas-merge-how-to-avoid-duplicating-columns

# join fractional difference 
simulation_results = pd.merge(simulation_results, scenario_fractional_differences, how='left', left_on=unique_cols, right_on=unique_cols, 
                                  left_index=False, right_index=False, sort=False, suffixes=('', '_y'), copy=None, indicator=False, validate=None)

simulation_results = simulation_results.drop(simulation_results.filter(regex='_y$').columns, axis=1)  # https://stackoverflow.com/questions/19125091/pandas-merge-how-to-avoid-duplicating-columns

# join AC demand data onto results 
simulation_results = pd.merge(simulation_results, differences_with_ac_demand, how='left', left_on=unique_cols, right_on=unique_cols, 
                                  left_index=False, right_index=False, sort=False, suffixes=('', '_y'), copy=None, indicator=False, validate=None)

simulation_results = simulation_results.drop(simulation_results.filter(regex='_y$').columns, axis=1)  # https://stackoverflow.com/questions/19125091/pandas-merge-how-to-avoid-duplicating-columns

# del scenario_fractional_differences, scenario_differences

## Rename desired columns

### Categorize the season of each data point

In [43]:
def seasonalize(df):
    # Map the month number to the season 
    season_number_to_name = {
        '1': 'Winter',
        '2': 'Spring',
        '3': 'Summer',
        '4': 'Fall', 
        }

    df['Season'] = df['Month']%12 // 3 + 1     # convert month number to season number 
    df['Season'] = df['Season'].astype(str).replace(season_number_to_name)
    return df

simulation_results = seasonalize(simulation_results)

### Calculate magnitude cols for total energy burden (changes)

In [44]:
# energy_burden_cols = ['Energy Burden - High [Monthly]',
#                       'Abs diff in Energy Burden - High [Monthly]',
#                       'Frac diff in Energy Burden - High [Monthly]',
#                       ]

# for col in energy_burden_cols: 
#     if col not in simulation_results.columns: 
#         print(col)

#     simulation_results[col + ' Magnitude'] = simulation_results[col].abs()

Rename columns

In [ ]:
default_column_mapping(simulation_results)

In [ ]:
column_renaming = {
	'Weather Scenario': 'Climate Change Scenario',
    
	'in.ashrae_iecc_climate_zone_2004': 'ASHRAE IECC Climate Zone 2004',
    'in.building_america_climate_zone': 'Building America Climate Zone',
	'in.city': 'City',
    'in.state': 'State',
	'in.federal_poverty_level': 'Federal Poverty Level',
	'in.geometry_building_type_recs': 'RECS Building Type',
	'in.income': 'Income',    
	'in.infiltration': 'Infiltration',
	'in.heating_fuel': 'Heating Fuel',
    'in.hvac_cooling_type': 'HVAC Cooling Type',
    
	'Energy Burden - High [Monthly]': 'Energy Burden',
	'Frac diff in Energy Burden - High [Monthly]': 'Frac diff in Energy Burden',
	'Abs diff in Energy Burden - High [Monthly]': 'Abs diff in Energy Burden',
	}

_ = return_modified_headers(column_renaming)

In [47]:
simulation_results = simulation_results.rename(columns=column_renaming, errors="raise")

# Save results

In [48]:
output_folder = Path("/Users/camilotoruno/Documents/local_research_data/simulations 24.08.12/results_old_method")
if not output_folder.exists(): output_folder.mkdir(parents=True, exist_ok=True)

filename = Path(output_folder, "simulations_results")
simulation_results.to_csv(str(filename) + ".csv", index=False)
simulation_results.to_csv(str(filename) + ".zip", compression={'method': 'zip', 'archive_name': "simulations_results.csv"}, index=False)